# Assignment: When Chat Systems Break - A Realistic Sharding Simulation
**Student:** Jensen Huang  
**Deadline:** 11th April 2026  
**Professor:** Prof. Poonam Khanvilkar

---

## 📅 Day 1–2 (2 April): System Thinking — Where Will the System Fail?

### Situation:
- 50,000 users join in 5 minutes
- 1 channel gets 80% of traffic (e.g., Cricket Final live chat)

### Q: If we had only ONE server — what fails first?

### Answer:

**1. Memory (RAM) fills up first**  
Every message is stored in a Python list on that one server. If 50,000 users each send 10 messages, that's 500,000 messages in RAM. The server runs out of memory.

**2. The server becomes too slow (CPU overload)**  
When thousands of messages come in every second, the server CPU is busy handling all of them. New messages start getting delayed or dropped.

**3. The one hot channel becomes a bottleneck**  
80% of messages going to one channel means that channel's data is being written constantly. Everything slows down because of this one channel.

### What data grows fastest?
- **Messages list** — grows with every single message sent
- **Channel 1's messages** — grows 4x faster than all other channels combined (80% traffic)
- **User connections** — 50,000 users joining = 50,000 open connections to manage

### Where are the bottlenecks?
| Bottleneck | Why it happens |
|---|---|
| Single message list | No separation, everything piles up together |
| One hot channel | 80% load on one place |
| Single server RAM | No way to distribute memory |
| Single server CPU | Cannot handle 1000s of requests per second |


---
## 📅 Day 3–4 (3 & 4 April): Basic Chat System + Load Simulation

In [ ]:
# =============================================
# DAY 3: Build a Basic (Naive) Chat System
# One server, stores everything in one list
# =============================================

import time
import random

# A single message with user, channel, and content
class Message:
    def __init__(self, user_id, channel_id, content):
        self.user_id = user_id
        self.channel_id = channel_id
        self.content = content

# One server that stores ALL messages in one list
class ChatServer:
    def __init__(self):
        self.messages = []  # everything goes here

    def send_message(self, message):
        self.messages.append(message)

    def stats(self):
        print("Total messages:", len(self.messages))


# --- Test with 10 users (small load - should work fine) ---
print("=== Small Load: 10 users, 100 messages ===")
server = ChatServer()

start = time.time()
for i in range(100):
    msg = Message(user_id=random.randint(1, 10), channel_id=1, content="hello")
    server.send_message(msg)
end = time.time()

server.stats()
print(f"Time taken: {round(end - start, 4)} seconds")
print("Result: Works fine!")

In [ ]:
# =============================================
# DAY 4: Simulate Large Load (10,000 messages)
# Observe: same server, now under heavy load
# =============================================

print("=== Large Load: 10,000 users, 50,000 messages ===")
server2 = ChatServer()

start = time.time()
for i in range(50000):
    msg = Message(user_id=random.randint(1, 10000), channel_id=random.randint(1, 50), content="hello")
    server2.send_message(msg)
end = time.time()

server2.stats()
print(f"Time taken: {round(end - start, 4)} seconds")
print()
print("--- Observation ---")
print("All 50,000 messages are stored in ONE list on ONE server.")
print("Problems with this approach:")
print("  1. Memory: All messages in RAM, no limit")
print("  2. No separation by channel or user")
print("  3. If this server crashes, ALL data is lost")
print("  4. Cannot scale — only 1 machine doing all the work")

---
## 📅 Day 5 (5 April): Introduce Shards — Split Into Multiple Servers

In [ ]:
# =============================================
# DAY 5: Shards — 3 separate mini-servers
# Each shard is independent (no global storage)
# =============================================

# A Shard = one mini server with its own message list
class Shard:
    def __init__(self, shard_id):
        self.id = shard_id
        self.messages = []  # only this shard's messages

    def store(self, message):
        self.messages.append(message)

    def count(self):
        return len(self.messages)


# ShardManager = the router that decides which shard to use
class ShardManager:
    def __init__(self, num_shards):
        # Create 3 independent shards
        self.shards = [Shard(i) for i in range(num_shards)]

    def send_message(self, message):
        # We will fill this logic in the next days
        pass

    def print_stats(self):
        total = sum(s.count() for s in self.shards)
        print(f"{'Shard':<10} {'Messages':<15} {'% of Total':<10}")
        print("-" * 35)
        for shard in self.shards:
            pct = round((shard.count() / total * 100), 1) if total > 0 else 0
            print(f"Shard {shard.id:<5} {shard.count():<15} {pct}%")
        print(f"{'Total':<10} {total}")


# Quick test: just create shards and verify they are independent
manager = ShardManager(num_shards=3)
print("Created 3 independent shards:")
for shard in manager.shards:
    print(f"  Shard {shard.id} — messages: {shard.count()}")
print()
print("Each shard has its OWN messages list (no global storage). ✓")

---
## 📅 Day 6 (6 April): User-Based Sharding — First Wrong Decision

In [ ]:
# =============================================
# DAY 6: User-Based Sharding
# Logic: user_id % num_shards → decides which shard
# Problem: one popular user floods one shard
# =============================================

class UserShardManager(ShardManager):

    def get_shard(self, user_id):
        # user 0,3,6 → shard 0 | user 1,4,7 → shard 1 | user 2,5,8 → shard 2
        return self.shards[user_id % len(self.shards)]

    def send_message(self, message):
        shard = self.get_shard(message.user_id)
        shard.store(message)


# --- Simulate: 1 influencer (user_id=1) sends 5000 messages ---
# --- Other 99 normal users send a few messages each ---

print("=== User-Based Sharding Simulation ===")
print("Scenario: 1 influencer (user 1) sends 5000 messages")
print("          99 normal users send ~10 messages each")
print()

user_manager = UserShardManager(num_shards=3)

# Influencer sends 5000 messages
for i in range(5000):
    msg = Message(user_id=1, channel_id=random.randint(1, 10), content="watch my stream!")
    user_manager.send_message(msg)

# 99 normal users each send ~10 messages
for user in range(2, 101):
    for _ in range(10):
        msg = Message(user_id=user, channel_id=random.randint(1, 10), content="hello")
        user_manager.send_message(msg)

print("--- Messages per Shard ---")
user_manager.print_stats()
print()
print("--- Observation ---")
print("User 1 → user_id 1 % 3 = Shard 1")
print("Shard 1 is OVERLOADED with 5000 messages from just 1 user!")
print("Shards 0 and 2 are mostly idle.")
print("This is called LOAD IMBALANCE — one shard doing all the work.")

---
## 📅 Day 7 (7 April): Channel-Based Sharding — Second Wrong Decision

In [ ]:
# =============================================
# DAY 7: Channel-Based Sharding
# Logic: channel_id % num_shards → decides shard
# Problem: viral channel → one shard gets all traffic
# =============================================

class ChannelShardManager(ShardManager):

    def get_shard(self, channel_id):
        return self.shards[channel_id % len(self.shards)]

    def send_message(self, message):
        shard = self.get_shard(message.channel_id)
        shard.store(message)


# --- Simulate: channel 3 (Cricket Final) goes viral ---
# --- 80% of all messages go to channel 3 ---

print("=== Channel-Based Sharding Simulation ===")
print("Scenario: Channel 3 (Cricket Final) goes viral")
print("          80% of messages go to channel 3")
print("          20% spread across other channels")
print()

ch_manager = ChannelShardManager(num_shards=3)

total_messages = 10000
viral_messages = int(total_messages * 0.8)  # 8000 to channel 3
normal_messages = total_messages - viral_messages  # 2000 spread

# 8000 messages to the viral channel (channel_id = 3)
for i in range(viral_messages):
    msg = Message(user_id=random.randint(1, 50000), channel_id=3, content="INDIA WINS!")
    ch_manager.send_message(msg)

# 2000 messages spread across other channels (not 3)
other_channels = [1, 2, 4, 5, 6, 7, 8, 9, 10]
for i in range(normal_messages):
    ch_id = random.choice(other_channels)
    msg = Message(user_id=random.randint(1, 50000), channel_id=ch_id, content="hello")
    ch_manager.send_message(msg)

print("--- Messages per Shard ---")
ch_manager.print_stats()
print()
print("--- Observation ---")
print(f"channel_id 3 % 3 = 0 → ALL viral messages go to Shard 0!")
print("Shard 0 = OVERLOADED (hotspot problem)")
print("Shard 1 and Shard 2 = mostly idle")
print()
print("--- Comparison with User-Based Sharding ---")
print("User-based: 1 active user floods 1 shard")
print("Channel-based: 1 viral channel floods 1 shard")
print("Both strategies fail during spikes — just for different reasons!")

---
## 📅 Day 8 (8 April): Hash-Based Sharding — Better but Not Perfect

In [ ]:
# =============================================
# DAY 8: Hash-Based Sharding
# Use MD5 hash to distribute more evenly
# Decision: what do we hash? user_id? channel_id? message_id?
# =============================================

import hashlib

class HashShardManager(ShardManager):

    def get_shard(self, key):
        # MD5 hash gives a big number, we use % to pick a shard
        h = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
        return self.shards[h % len(self.shards)]

    def send_message(self, message):
        # DECISION: We hash channel_id
        # Why? Because we want messages of the same channel together,
        # and hash spreads channels more evenly than simple modulo.
        shard = self.get_shard(message.channel_id)
        shard.store(message)


# --- Same viral scenario as Day 7 ---
print("=== Hash-Based Sharding Simulation ===")
print("Same scenario: Channel 3 (Cricket Final) is viral (80% traffic)")
print("Key choice: hashing channel_id")
print()

hash_manager = HashShardManager(num_shards=3)

for i in range(8000):
    msg = Message(user_id=random.randint(1, 50000), channel_id=3, content="INDIA WINS!")
    hash_manager.send_message(msg)

for i in range(2000):
    ch_id = random.choice([1, 2, 4, 5, 6, 7, 8, 9, 10])
    msg = Message(user_id=random.randint(1, 50000), channel_id=ch_id, content="hello")
    hash_manager.send_message(msg)

print("--- Messages per Shard ---")
hash_manager.print_stats()
print()
print("--- Observation ---")
print("Hash of channel_id=3 still maps to ONE shard.")
print("So hash-based sharding does NOT solve the hotspot problem")
print("when ONE channel dominates traffic.")
print()
print("--- Why Hash? What's better about it? ---")
print("If traffic is SPREAD across many channels, hash gives better distribution.")
print("Example: channels 1-50 each with equal traffic → hash spreads them evenly.")
print()
print("--- Key Choice Explanation ---")
print("Hash user_id   → same problem as Day 6 (one user = one shard)")
print("Hash channel_id → channel messages stay together (good for reads)")
print("Hash message_id → very even spread, but queries are harder (Day 10 problem)")
print("We chose channel_id as it keeps related messages in one place.")
print()
print("--- The Big Limitation: Shard Count Changes ---")
print("If we had 3 shards and now add 1 more (total 4):")
print("  Old: hash(channel_3) % 3 = some shard")
print("  New: hash(channel_3) % 4 = DIFFERENT shard")
print("  All existing data is now in the WRONG shard! System breaks.")

---
## 📅 Day 9 (9 April): Stress Simulation + Failure Simulation

In [ ]:
# =============================================
# DAY 9: Stress Simulation
# Run all 3 strategies under 3 scenarios:
#   1. Normal day
#   2. Viral event
#   3. Extreme spike
# Also: Hotspot detection, Failure simulation
# =============================================

def simulate(manager, num_users, num_messages, viral_channel=None, viral_ratio=0.0):
    """
    Sends messages to a shard manager.
    viral_channel: if set, this channel gets viral_ratio % of traffic
    """
    viral_count = int(num_messages * viral_ratio)
    normal_count = num_messages - viral_count

    # Viral messages
    if viral_channel:
        for _ in range(viral_count):
            msg = Message(user_id=random.randint(1, num_users),
                          channel_id=viral_channel,
                          content="hello")
            manager.send_message(msg)

    # Normal messages
    for _ in range(normal_count):
        msg = Message(user_id=random.randint(1, num_users),
                      channel_id=random.randint(1, 50),
                      content="hello")
        manager.send_message(msg)


def check_hotspot(manager, threshold=50):
    """Warn if any shard has more than threshold% of total messages"""
    total = sum(s.count() for s in manager.shards)
    if total == 0:
        return
    for shard in manager.shards:
        pct = shard.count() / total * 100
        if pct > threshold:
            print(f"  ⚠️  HOTSPOT WARNING: Shard {shard.id} has {round(pct,1)}% of load! (>{threshold}% threshold)")


scenarios = [
    {"name": "Normal Day",     "users": 1000, "messages": 5000,  "viral_ch": None, "viral_ratio": 0.0},
    {"name": "Viral Event",    "users": 5000, "messages": 20000, "viral_ch": 3,    "viral_ratio": 0.8},
    {"name": "Extreme Spike",  "users": 50000,"messages": 50000, "viral_ch": 3,    "viral_ratio": 0.95},
]

strategies = [
    ("User-Based",    UserShardManager),
    ("Channel-Based", ChannelShardManager),
    ("Hash-Based",    HashShardManager),
]

for scenario in scenarios:
    print(f"\n{'='*55}")
    print(f"SCENARIO: {scenario['name']}")
    print(f"  Users: {scenario['users']} | Messages: {scenario['messages']}")
    if scenario['viral_ch']:
        print(f"  Viral Channel: {scenario['viral_ch']} ({int(scenario['viral_ratio']*100)}% of traffic)")
    print(f"{'='*55}")

    for strategy_name, ManagerClass in strategies:
        mgr = ManagerClass(num_shards=3)
        simulate(mgr,
                 num_users=scenario['users'],
                 num_messages=scenario['messages'],
                 viral_channel=scenario['viral_ch'],
                 viral_ratio=scenario['viral_ratio'])

        print(f"\n  Strategy: {strategy_name}")
        total = sum(s.count() for s in mgr.shards)
        for shard in mgr.shards:
            pct = round(shard.count() / total * 100, 1) if total > 0 else 0
            bar = '█' * int(pct / 5)
            print(f"    Shard {shard.id}: {shard.count():>6} msgs ({pct}%) {bar}")
        check_hotspot(mgr, threshold=50)

In [ ]:
# =============================================
# DAY 9 (continued): Shard Failure Simulation
# Disable one shard and observe what happens
# =============================================

print("=== Failure Simulation: What happens when Shard 0 goes down? ===")
print()

class ShardWithFailure(Shard):
    def __init__(self, shard_id):
        super().__init__(shard_id)
        self.is_down = False   # by default it's working

    def store(self, message):
        if self.is_down:
            # Message is LOST — shard is down
            return False
        self.messages.append(message)
        return True


class FailureChannelManager(ChannelShardManager):
    def __init__(self, num_shards):
        # Override to use failure-aware shards
        self.shards = [ShardWithFailure(i) for i in range(num_shards)]
        self.lost_messages = 0

    def send_message(self, message):
        shard = self.get_shard(message.channel_id)
        success = shard.store(message)
        if not success:
            self.lost_messages += 1


fail_mgr = FailureChannelManager(num_shards=3)

# Send 3000 messages (normal)
for i in range(3000):
    msg = Message(user_id=random.randint(1, 1000),
                  channel_id=random.randint(1, 9),
                  content="hello")
    fail_mgr.send_message(msg)

print("Before failure:")
for shard in fail_mgr.shards:
    print(f"  Shard {shard.id}: {shard.count()} messages, Status: {'✓ UP' if not shard.is_down else '✗ DOWN'}")

# NOW: Shard 0 goes down!
print()
print(">>> Shard 0 has crashed! <<<")
fail_mgr.shards[0].is_down = True

# Send 2000 more messages
for i in range(2000):
    msg = Message(user_id=random.randint(1, 1000),
                  channel_id=random.randint(1, 9),
                  content="hello after crash")
    fail_mgr.send_message(msg)

print()
print("After failure:")
for shard in fail_mgr.shards:
    print(f"  Shard {shard.id}: {shard.count()} messages, Status: {'✓ UP' if not shard.is_down else '✗ DOWN'}")

print()
print(f"Messages LOST because Shard 0 was down: {fail_mgr.lost_messages}")
print("Observation: Any channel that maps to Shard 0 lost all its new messages.")
print("In a real system, we'd need REPLICATION (backup copies) to prevent data loss.")

---
## 📅 Day 10 (Final Analysis): Cross-Shard Query + System Evolution

In [ ]:
# =============================================
# DAY 10: Cross-Shard Query
# Task: Fetch last 10 messages of a channel
# Problem: Messages might be spread across shards
# =============================================

print("=== Cross-Shard Query: Fetch last 10 messages of a channel ===")
print()

# Setup: Use hash-based manager with some data
cross_mgr = HashShardManager(num_shards=3)

# Add messages with message IDs so we can sort them
msg_id = 0
for i in range(500):
    ch = random.randint(1, 10)
    msg = Message(user_id=random.randint(1, 100), channel_id=ch, content=f"msg-{msg_id}")
    msg.msg_id = msg_id   # add a simple message ID
    msg_id += 1
    cross_mgr.send_message(msg)


def fetch_last_n_from_channel(manager, channel_id, n=10):
    """
    Fetch last n messages of a channel.
    We have to check ALL shards because we don't know where data is.
    (In hash-based, channel maps to 1 shard. But in general, we check all.)
    """
    shards_checked = 0
    all_channel_msgs = []

    for shard in manager.shards:
        shards_checked += 1
        for msg in shard.messages:
            if msg.channel_id == channel_id:
                all_channel_msgs.append(msg)

    # Sort by msg_id to get order (latest = highest id)
    all_channel_msgs.sort(key=lambda m: m.msg_id, reverse=True)

    print(f"  Shards checked to find channel {channel_id}: {shards_checked}")
    print(f"  Total messages found for channel {channel_id}: {len(all_channel_msgs)}")
    print(f"  Last {n} messages (most recent first):")
    for m in all_channel_msgs[:n]:
        print(f"    [msg_id={m.msg_id}] User {m.user_id}: {m.content}")


fetch_last_n_from_channel(cross_mgr, channel_id=3, n=10)
print()
print("Observation: We had to scan ALL shards to find the messages.")
print("Cost: If 10 shards, we check 10 shards for every read query.")
print("This is why READ queries are harder than WRITE in sharded systems.")

In [ ]:
# =============================================
# DAY 10: System Evolution — 3 shards → 6 shards
# What breaks when shard count changes?
# =============================================

print("=== System Evolution: What happens when shards increase from 3 → 6? ===")
print()

import hashlib

def shard_for_channel(channel_id, num_shards):
    h = int(hashlib.md5(str(channel_id).encode()).hexdigest(), 16)
    return h % num_shards

print("Channel mapping BEFORE (3 shards):")
for ch in range(1, 11):
    print(f"  Channel {ch:>2} → Shard {shard_for_channel(ch, 3)}")

print()
print("Channel mapping AFTER adding more shards (6 shards):")
moved = 0
for ch in range(1, 11):
    old = shard_for_channel(ch, 3)
    new = shard_for_channel(ch, 6)
    flag = "  ← MOVED!" if old != new else ""
    if old != new:
        moved += 1
    print(f"  Channel {ch:>2} → was Shard {old}, now Shard {new}{flag}")

print()
print(f"Channels whose shard location CHANGED: {moved} out of 10")
print("Problem: Old data is still in the OLD shard. New messages go to NEW shard.")
print("Result: Queries find INCOMPLETE data — system is BROKEN!")
print("Solution (not implemented here): Consistent Hashing or manual data migration.")

In [ ]:
# =============================================
# DAY 10: Final Analysis — Answers to all questions
# =============================================

print("=" * 60)
print("FINAL ANALYSIS")
print("=" * 60)

print("""
Q1. Which shard failed first and why?
    In Channel-Based sharding:
    → Shard 0 (channel_id=3 % 3 = 0) failed first.
    → Because the viral cricket channel (channel 3) sent 80-95%
      of all traffic to shard 0, overloading it completely.
    In User-Based sharding:
    → Shard 1 (user_id=1 % 3 = 1) failed first.
    → Because the influencer (user 1) sent 5000 messages alone.

Q2. Which strategy looked good but failed under spike?
    → Channel-Based sharding looked logical at first.
    → Keeping channel messages together makes reads easier.
    → BUT during a viral event (1 channel dominates), it
      becomes a hotspot and fails completely.

Q3. What happens if shards increase from 3 → 10?
    → Hash functions use % num_shards.
    → When num_shards changes, almost all channel-to-shard
      mappings change.
    → Old data stays in old shards. New data goes to new shards.
    → Queries return INCOMPLETE results — system is broken.
    → This is why scaling up shards is very hard in practice.

Q4. What breaks when one shard goes down?
    → All messages that were going to that shard are LOST.
    → Any channel mapped to that shard becomes UNREADABLE.
    → Without replication (backup copies), data is gone forever.
    → This is why real systems like Discord use replication.
"""
)

print("=" * 60)
print("STRATEGY COMPARISON SUMMARY")
print("=" * 60)
print(f"{'Strategy':<20} {'Works Well When':<30} {'Fails When'}")
print("-" * 70)
print(f"{'User-Based':<20} {'Many equal-activity users':<30} {'One user is very active'}")
print(f"{'Channel-Based':<20} {'Many equal-traffic channels':<30} {'One channel goes viral'}")
print(f"{'Hash-Based':<20} {'Load spread across many keys':<30} {'One key dominates OR shard count changes'}")
print()
print("Conclusion: No single strategy is perfect.")
print("Real systems combine multiple approaches + replication + monitoring.")